# API 设计


## API 是什么

在浏览器里打开一个网页，浏览器会调用十几个 API：拿用户信息、拿商品列表、拿评论、下单。这些 API 的风格有几种——REST、GraphQL、gRPC。

## REST

REST 是目前最主流的 API 设计风格。核心思想：把一切看成资源，用 HTTP 方法操作它们。

```
GET    /users          # 获取用户列表
GET    /users/123      # 获取用户 123
POST   /users          # 新建用户
PUT    /users/123      # 完整替换用户 123
PATCH  /users/123      # 部分修改用户 123（如只改名字）
DELETE /users/123      # 删除用户 123
```

REST 的问题：前端调用 `/users` 拿到整个用户对象（包括不需要的字段），再调 `/users/123/orders` 拿订单，再调 `/orders/456/items` 拿商品——一次页面加载可能涉及十个请求。这叫做过度获取和请求瀑布。

## GraphQL

GraphQL 解决了 REST 的两个核心问题：一次请求拿刚好需要的数据，不多不少。

它不是换 URL，是换思想——不再暴露多个端点（`/users`、`/orders`），只有**一个端点**（`/graphql`），前端自己用查询语言声明想要什么：

```graphql
query {
  user(id: 123) {
    name
    email
    orders {       # 一次请求，连用户带订单一起拿
      total
      date
    }
  }
}
```

后端只返回你声明了的字段，没有多余数据。前端不用等多次请求——一次搞定。

| | REST | GraphQL |
|------|------|------|
| 端点数量 | 多个，每个资源一个 | 一个 `/graphql` |
| 数据量 | 后端决定返回什么，可能过多或过少 | 前端精确声明要什么 |
| 学习成本 | 低 | 中——需要学查询语法 |
| 缓存 | 简单，利用 HTTP 缓存 | 复杂，所有请求走同一个 URL |
| 适合场景 | 简单 CRUD 应用 | 复杂数据关系的应用 |

GraphQL 不是 REST 的替代品——复杂应用用它省请求数，简单应用用它反而增加复杂度。


## gRPC

REST 和 GraphQL 都用 JSON 传数据——人可读，但慢。gRPC 用 Protocol Buffers（protobuf）——二进制格式，小、快，但人读不了。

```protobuf
// 定义消息格式
message User {
  int32 id = 1;
  string name = 2;
  string email = 3;
}

// 定义方法
service UserService {
  rpc GetUser (UserId) returns (User) {}
}
```

protobuf 文件是强类型的——写错一个字段编译阶段就报错，不像 JSON 拼错一个 key 到线上才发现。

| | REST/GraphQL | gRPC |
|------|-------------|------|
| 数据格式 | JSON（文本） | Protobuf（二进制） |
| 速度 | 慢（序列化开销 + 带宽） | 快（二进制，体积小 3-10 倍） |
| 可读性 | 人眼能看 | 需要工具 |
| 类型安全 | 弱（JSON 无类型约束） | 强（proto 文件定义清楚） |
| 适用 | 浏览器 ↔ 服务器、公开 API | 微服务之间、手机 App ↔ 服务器 |

> gRPC 跑在 HTTP/2 上，天然支持双向流——不只是请求-响应，可以服务端主动推送、客户端和服务端同时发数据。


## API 版本化

上线后你的 API 不能随便改——别人已经照着你的接口写了代码。改了字段名或删了旧字段，别人 App 立刻崩。版本化让你能推进新功能的同时不破坏旧用户。

三种做法：

| 方式 | 怎么做 | 优缺点 |
|------|------|------|
| URL 版本 | `/v1/users` → `/v2/users` | 直观，但 URL 变脏 |
| 请求头 | 在请求头里标版本 | URL 干净，但不够直观 |
| 向后兼容 | 只加新字段不删旧字段，永不升版本 | 最理想，但实际做不到 |

实践中 URL 版本最常见。调 API 时多看一眼前面的 `v1`——那是版本号。


## OpenAPI / Swagger

API 设计好了，怎么让别人知道怎么调？口头说明靠不住、手写文档没人维护。OpenAPI（原 Swagger）用一份 YAML 或 JSON 文件精确描述 API，机器可读、人可以看。

```yaml
# OpenAPI 片段
/users/{id}:
  get:
    summary: 获取用户
    parameters:
      - name: id
        in: path
        required: true
        schema:
          type: integer
    responses:
      '200':
        description: 成功
```

有这份描述，自动生成文档页面，自动生成前端和后端的调用代码，自动校验请求格式——代码即文档，不会过期。


## Webhook：服务器主动通知

前面的都是前端主动调用后端。Webhook 反过来——后端主动通知前端。

```
你下单 → 支付宝 → 支付完成 → 支付宝发 POST 到你的 /payment-callback
                         （URL 是你提前告诉它的）
```

Webhook 注册一个 URL，告诉对方"有事发生就往这个地址 POST"。支付宝不会每分钟轮询银行"钱到了没"——银行收到了直接通知支付宝。这就是 Webhook。
